In [6]:
import tkinter as tk
#from tkinter import filedialog
import time
import pandas as pd
import numpy as np
#!pip install matplotlib
import matplotlib.pyplot as plt
from tkinter import messagebox
import threading
import os

In [7]:
#excel datei importieren


In [8]:
def EVO_Read_Input_from_ExcelFile(self):

    # Initialisierung
    self.P_PVspez = None
    self.P_PV_Steps = None
    self.P_PV = None
    self.P_Load = None
    self.P_Batt_Steps = None
    self.E_Batt_Steps = None
    self.P_BATTMAX = None
    self.E_BATTMAX = None
    self.Eta_Batt = None
    self.Eta_WRBatt = None
    self.SCHRITT = None

    #  Einlesen Excel-Datei
    self.Path = self.PathEntry.get()
    self.df = pd.read_excel(self.Path, sheet_name='SimBatt-Input')

    #  Auslesen relevanter Serien aus df und Konvertierung zu np array
    self.P_PVspez = pd.Series(self.df.loc[0:35040, 'PV-Erzeugungsprofil in kW/kWp'])
    self.P_PVspez = self.P_PVspez.to_numpy()
    self.P_Load = pd.Series(self.df.loc[0:35040, 'Last in kWh'])#
    self.P_Load = self.P_Load.to_numpy()
    self.P_PV_Steps = pd.Series(self.df.loc[0:9,'PV: Leistungsschritte in kWp'])
    self.P_PV_Steps = self.P_PV_Steps.to_numpy()
    self.E_Batt_Steps = pd.Series(self.df.loc[0:9,'Speicher: Kapazität in kWh'])
    self.E_Batt_Steps = self.E_Batt_Steps.to_numpy()
    self.P_Batt_Steps = pd.Series(self.df.loc[0:9,'Speicher: Leistung in kW'])
    self.P_Batt_Steps = self.P_Batt_Steps.to_numpy()
    self.Eta_Batt = pd.Series(self.df.at[0, 'Speicher: Wirkungsgrad in %'])
    self.Eta_Batt = self.Eta_Batt.to_numpy()
    self.Eta_WRBatt = pd.Series(self.df.at[0, 'Speicher WR: Wirkungsgrad in %'])
    self.Eta_WRBatt = self.Eta_WRBatt.to_numpy()
    self.SCHRITT = pd.Series(self.df.at[0, 'Schrittweite'])
    self.SCHRITT = self.SCHRITT.to_numpy()

In [ ]:
def SimPVdirekt(self):

    # Initialisierung Variable
    self.P_PVdirekt = np.zeros(len(self.P_Load))
    
    #  Startzeit-Endzeit-Schrittweite
    self.t_START = 0
    self.T_END = len(self.P_Load)

    # PV-Direktverbrauch berechnen
    for t in range(self.t_START,self.T_END):
        if self.P_PV[t] > 0:
            self.P_PVdirekt[t] = min(self.P_PV[t],self.P_Load[t])
        else:
            self.P_PVdirekt[t] = 0


In [ ]:
def SimBatEVO(self):

    #  Differenzleistung
    self.P_DIFF = self.P_PV - self.P_Load

    #  Startzeit-Endzeit-Schrittweite
    self.t_START = 0
    self.T_END = len(self.P_Load)

    # Vorinitialisierung benötigter Variablen
    self.P_BATTIN = np.zeros(len(self.P_Load))
    self.P_BATTOUT = np.zeros(len(self.P_Load)) 
    self.P_BATT = np.zeros(len(self.P_Load))
    self.P_BS = np.zeros(len(self.P_Load)) 
    self.E_BATT = np.zeros(len(self.P_Load))
    self.SOC = np.zeros(len(self.P_Load))
    self.P_Netz = np.zeros(len(self.P_Load))


    #  PV-Speichersimulation
    for t in range(self.t_START, self.T_END):
        
        #  Batterieladung
        if self.P_DIFF[t] > 0:
            self.P_BATTIN[t] = min((min(self.P_BATTMAX, self.P_DIFF[t])*self.Eta_WRBatt), (self.E_BATTMAX*(1-self.SOC[t-1])/self.SCHRITT/self.Eta_Batt))
        #Batterieentladung
        elif  self.P_DIFF[t] < 0:
            self.P_BATTOUT[t] = -min((min(self.P_BATTMAX, -self.P_DIFF[t])/self.Eta_WRBatt), (self.E_BATTMAX*self.SOC[t-1]/self.SCHRITT))

        # Batteriespeicherleistung
        self.P_BATT[t] = self.P_BATTIN[t]+self.P_BATTOUT[t]

        #  Batteriesystemleistung
        self.P_BS[t] = self.P_BATTIN[t]/self.Eta_WRBatt + self.P_BATTOUT[t]*self.Eta_WRBatt

        #  Gespeicherte Energie
        self.E_BATT[t] = self.E_BATT[t-1] + (self.P_BATTIN[t]*self.Eta_Batt + self.P_BATTOUT[t])*self.SCHRITT

        #  SOC bestimmen
        self.SOC[t] = self.E_BATT[t]/self.E_BATTMAX

In [ ]:
def SaveSimBat(self):
    self.P_PV_BS_Variation_trans = np.transpose(self.P_PV_BS_Variation)
    self.df2Excel = pd.DataFrame(self.P_PV_BS_Variation_trans)
    with pd.ExcelWriter(self.Path, mode='a', if_sheet_exists='replace', engine='openpyxl') as writer:
        self.df2Excel.to_excel(writer, sheet_name='SimBatt-Output')
